# ECAPA-TDNN Fine-Tuning on Custom Voice Dataset

**What this does**: Fine-tunes the pre-trained ECAPA-TDNN model on YOUR 25-speaker dataset using Triplet Loss.

**Steps**:
1. Install dependencies & upload dataset
2. Convert OGG to WAV, create train/val split
3. Fine-tune with Triplet Loss (20 epochs)
4. Evaluate before vs after
5. Download the fine-tuned model

> **Runtime**: Set to **GPU** (Runtime > Change runtime type > T4 GPU)

## 1. Install Dependencies

In [ ]:
!pip install -q speechbrain torch torchaudio soundfile librosa matplotlib scikit-learn numpy

## 2. Upload & Prepare Dataset

In [ ]:
import os, zipfile, csv, random
import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
import torchaudio
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

if not hasattr(torchaudio, 'list_audio_backends'):
    torchaudio.list_audio_backends = lambda: ['soundfile']

from speechbrain.inference.speaker import EncoderClassifier
print('All imports OK')

In [ ]:
from google.colab import files
print('Upload voice-set.zip...')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')
print('Extracted!')

## 3. Convert OGG to WAV & Create Split

In [ ]:
EXTRACT_DIR = 'voice-set'
OUTPUT_DIR = 'dataset_processed'
SAMPLE_RATE = 16000
os.makedirs(OUTPUT_DIR, exist_ok=True)

speakers = sorted([d for d in os.listdir(EXTRACT_DIR)
                    if os.path.isdir(os.path.join(EXTRACT_DIR, d))])
speaker_to_id = {name: idx for idx, name in enumerate(speakers)}
print('Found %d speakers' % len(speakers))

train_data = defaultdict(list)  # speaker_id -> [wav_paths]
val_data = defaultdict(list)
total = 0

for speaker in speakers:
    sid = speaker_to_id[speaker]
    sp_dir = os.path.join(EXTRACT_DIR, speaker)
    out_dir = os.path.join(OUTPUT_DIR, 'speaker_%03d' % sid)
    os.makedirs(out_dir, exist_ok=True)
    oggs = sorted([f for f in os.listdir(sp_dir) if f.endswith('.ogg')])
    for ogg in oggs:
        cn = ogg.replace('.ogg', '')
        clip_num = int(cn) if cn.isdigit() else 0
        data, sr = sf.read(os.path.join(sp_dir, ogg))
        if len(data.shape) > 1:
            data = np.mean(data, axis=1)
        if sr != SAMPLE_RATE:
            import librosa
            data = librosa.resample(data, orig_sr=sr, target_sr=SAMPLE_RATE)
        if np.max(np.abs(data)) > 0:
            data = data / np.max(np.abs(data)) * 0.95
        wav_path = os.path.join(out_dir, 'clip_%d.wav' % clip_num)
        sf.write(wav_path, data, SAMPLE_RATE, subtype='PCM_16')
        if clip_num <= 5:
            train_data[sid].append(wav_path)
        else:
            val_data[sid].append(wav_path)
        total += 1
    print('  [%02d] %s: %d clips' % (sid, speaker, len(oggs)))

print('Total converted: %d' % total)
print('Train clips: %d | Val clips: %d' % (
    sum(len(v) for v in train_data.values()),
    sum(len(v) for v in val_data.values())))

## 4. Load Pre-trained ECAPA-TDNN

In [ ]:
print('Loading ECAPA-TDNN...')

import huggingface_hub
if not hasattr(huggingface_hub, '_original_download'):
    huggingface_hub._original_download = huggingface_hub.hf_hub_download
    def patched_download(*args, **kwargs):
        kwargs.pop('use_auth_token', None)
        return huggingface_hub._original_download(*args, **kwargs)
    huggingface_hub.hf_hub_download = patched_download

from speechbrain.inference.speaker import EncoderClassifier
model = EncoderClassifier.from_hparams(
    source='speechbrain/spkrec-ecapa-voxceleb',
    savedir='pretrained_models/spkrec-ecapa-voxceleb',
)
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
model.to(device)
total_params = sum(p.numel() for p in model.mods.embedding_model.parameters())
print('Loaded! Params:', format(total_params, ','), '| Embedding dim: 192')


## 5. Baseline Evaluation (Before Fine-Tuning)
Measure current performance to compare with after fine-tuning.

In [ ]:
def get_embedding_from_file(model, path, device):
    data, sr = sf.read(path)
    if len(data.shape) > 1:
        data = np.mean(data, axis=1)
    wavs = torch.FloatTensor(data).unsqueeze(0).to(device)
    wav_lens = torch.ones(1).to(device)
    with torch.no_grad():
        emb = model.encode_batch(wavs)
    return emb.squeeze().cpu()

def evaluate_model(model, val_data, device):
    model.mods.embedding_model.eval()
    embeddings = {}
    for sid, fpaths in val_data.items():
        for fp in fpaths:
            embeddings[(sid, fp)] = get_embedding_from_file(model, fp, device)
    cos = nn.CosineSimilarity(dim=0)
    same, diff = [], []
    keys = list(embeddings.keys())
    for i in range(len(keys)):
        for j in range(i+1, len(keys)):
            s = cos(embeddings[keys[i]], embeddings[keys[j]]).item()
            if keys[i][0] == keys[j][0]:
                same.append(s)
            else:
                diff.append(s)
    return np.mean(same), np.mean(diff)

print('Evaluating baseline...')
same_pre, diff_pre = evaluate_model(model, val_data, device)
print('BASELINE (before fine-tuning):')
print('  Same-speaker sim:  %.4f' % same_pre)
print('  Diff-speaker sim:  %.4f' % diff_pre)
print('  Gap:               %.4f' % (same_pre - diff_pre))

## 6. Define Triplet Dataset & Loss

In [ ]:
class TripletDataset(Dataset):
    def __init__(self, data_dict, max_len=48000):
        self.data = data_dict
        self.max_len = max_len
        self.sids = list(data_dict.keys())
        self.pairs = []
        for sid in self.sids:
            fps = data_dict[sid]
            for i in range(len(fps)):
                for j in range(i+1, len(fps)):
                    self.pairs.append((sid, fps[i], fps[j]))

    def __len__(self):
        return len(self.pairs)

    def _load(self, path):
        data, _ = sf.read(path)
        if len(data.shape) > 1:
            data = np.mean(data, axis=1)
        if len(data) < 16000:
            data = np.pad(data, (0, 16000 - len(data)))
        if len(data) > self.max_len:
            st = random.randint(0, len(data) - self.max_len)
            data = data[st:st+self.max_len]
        return torch.FloatTensor(data)

    def __getitem__(self, idx):
        sid, anc_path, pos_path = self.pairs[idx]
        neg_sid = random.choice([s for s in self.sids if s != sid])
        neg_path = random.choice(self.data[neg_sid])
        return self._load(anc_path), self._load(pos_path), self._load(neg_path), sid

def collate_triplets(batch):
    anchors, positives, negatives, sids = zip(*batch)
    max_len = max(max(a.shape[0] for a in anchors),
                  max(p.shape[0] for p in positives),
                  max(n.shape[0] for n in negatives))
    def pad(tensors):
        out = torch.zeros(len(tensors), max_len)
        lens = torch.zeros(len(tensors))
        for i, t in enumerate(tensors):
            out[i, :t.shape[0]] = t
            lens[i] = t.shape[0] / max_len
        return out, lens
    a, al = pad(anchors)
    p, pl = pad(positives)
    n, nl = pad(negatives)
    return a, al, p, pl, n, nl, torch.tensor(sids)

class TripletLoss(nn.Module):
    def __init__(self, margin=0.2):
        super().__init__()
        self.margin = margin
        self.cos = nn.CosineSimilarity(dim=1)
    def forward(self, anchor, positive, negative):
        pos_sim = self.cos(anchor, positive)
        neg_sim = self.cos(anchor, negative)
        loss = torch.clamp(self.margin - (pos_sim - neg_sim), min=0.0)
        return loss.mean(), pos_sim.mean().item(), neg_sim.mean().item()

print('Dataset & Loss defined!')

## 7. Fine-Tune the Model
This trains for 20 epochs. On GPU it takes ~5-10 minutes.

In [ ]:
# Freeze early layers, train later ones
ecapa = model.mods.embedding_model
for param in model.mods.compute_features.parameters():
    param.requires_grad = False
for param in model.mods.mean_var_norm.parameters():
    param.requires_grad = False

trainable_params = []
frozen = 0
trainable = 0
for name, param in ecapa.named_parameters():
    if name.startswith('blocks.0.') or name.startswith('blocks.1.'):
        param.requires_grad = False
        frozen += param.numel()
    else:
        param.requires_grad = True
        trainable_params.append(param)
        trainable += param.numel()

print('Frozen:', format(frozen, ','))
print('Trainable:', format(trainable, ','))

# Setup
EPOCHS = 20
BATCH_SIZE = 8
LR = 5e-5

dataset = TripletDataset(train_data)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                     num_workers=0, drop_last=True, collate_fn=collate_triplets)
optimizer = optim.Adam(trainable_params, lr=LR, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = TripletLoss(margin=0.2)

print('Training pairs:', len(dataset))
print('Batches/epoch:', len(loader))
print('Starting training...')
print('-' * 60)

best_gap = same_pre - diff_pre
history = {'loss': [], 'pos': [], 'neg': [], 'val_same': [], 'val_diff': []}

for epoch in range(1, EPOCHS + 1):
    ecapa.train()
    total_loss = 0
    total_pos = 0
    total_neg = 0
    nbatch = 0

    for a_wav, a_len, p_wav, p_len, n_wav, n_len, _ in loader:
        a_wav, a_len = a_wav.to(device), a_len.to(device)
        p_wav, p_len = p_wav.to(device), p_len.to(device)
        n_wav, n_len = n_wav.to(device), n_len.to(device)

        a_feat = model.mods.mean_var_norm(model.mods.compute_features(a_wav), a_len)
        p_feat = model.mods.mean_var_norm(model.mods.compute_features(p_wav), p_len)
        n_feat = model.mods.mean_var_norm(model.mods.compute_features(n_wav), n_len)

        a_emb = ecapa(a_feat, a_len).squeeze(1)
        p_emb = ecapa(p_feat, p_len).squeeze(1)
        n_emb = ecapa(n_feat, n_len).squeeze(1)

        loss, ps, ns = criterion(a_emb, p_emb, n_emb)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=5.0)
        optimizer.step()

        total_loss += loss.item()
        total_pos += ps
        total_neg += ns
        nbatch += 1

    scheduler.step()
    avg_loss = total_loss / max(nbatch, 1)
    avg_pos = total_pos / max(nbatch, 1)
    avg_neg = total_neg / max(nbatch, 1)

    history['loss'].append(avg_loss)
    history['pos'].append(avg_pos)
    history['neg'].append(avg_neg)

    if epoch % 5 == 0 or epoch == 1 or epoch == EPOCHS:
        vs, vd = evaluate_model(model, val_data, device)
        gap = vs - vd
        history['val_same'].append(vs)
        history['val_diff'].append(vd)
        tag = ' << BEST' if gap > best_gap else ''
        print('Epoch %02d/%d | Loss: %.4f | P/N: %.3f/%.3f | Val Same: %.3f Diff: %.3f Gap: %.3f%s' %
              (epoch, EPOCHS, avg_loss, avg_pos, avg_neg, vs, vd, gap, tag))
        if gap > best_gap:
            best_gap = gap
            os.makedirs('fine_tuned_model', exist_ok=True)
            torch.save(ecapa.state_dict(), 'fine_tuned_model/embedding_model.ckpt')
    else:
        print('Epoch %02d/%d | Loss: %.4f | pos: %.3f neg: %.3f' %
              (epoch, EPOCHS, avg_loss, avg_pos, avg_neg))

print('-' * 60)
print('Training complete!')

## 8. Training Progress Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Fine-Tuning Progress', fontsize=16, fontweight='bold')

ax = axes[0]
ax.plot(history['loss'], 'b-o', markersize=4)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Triplet Loss'); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(history['pos'], 'g-o', markersize=4, label='Positive (same)')
ax.plot(history['neg'], 'r-o', markersize=4, label='Negative (diff)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Similarity')
ax.set_title('Train Similarities'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2]
eval_epochs = [e for e in range(1, EPOCHS+1) if e % 5 == 0 or e == 1 or e == EPOCHS]
ax.plot(eval_epochs[:len(history['val_same'])], history['val_same'], 'g-s', markersize=6, label='Val Same')
ax.plot(eval_epochs[:len(history['val_diff'])], history['val_diff'], 'r-s', markersize=6, label='Val Diff')
ax.axhline(y=0.50, color='orange', linestyle='--', label='Threshold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Similarity')
ax.set_title('Validation Performance'); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_progress.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Final Evaluation: Before vs After

In [ ]:
print('Evaluating fine-tuned model...')
same_post, diff_post = evaluate_model(model, val_data, device)

print('=' * 55)
print('  FINE-TUNING RESULTS')
print('=' * 55)
print('  BEFORE:')
print('    Same-speaker:  %.4f' % same_pre)
print('    Diff-speaker:  %.4f' % diff_pre)
print('    Gap:           %.4f' % (same_pre - diff_pre))
print('  AFTER:')
print('    Same-speaker:  %.4f' % same_post)
print('    Diff-speaker:  %.4f' % diff_post)
print('    Gap:           %.4f' % (same_post - diff_post))
improvement = (same_post - diff_post) - (same_pre - diff_pre)
print('  IMPROVEMENT:     %+.4f' % improvement)
print('=' * 55)

# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(3)
w = 0.35
before = [same_pre, diff_pre, same_pre - diff_pre]
after = [same_post, diff_post, same_post - diff_post]
ax.bar(x - w/2, before, w, label='Before', color='#F44336', alpha=0.7)
ax.bar(x + w/2, after, w, label='After', color='#4CAF50', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(['Same-Speaker', 'Diff-Speaker', 'Gap'])
ax.set_ylabel('Cosine Similarity')
ax.set_title('Before vs After Fine-Tuning', fontsize=14, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
for i, (b, a) in enumerate(zip(before, after)):
    ax.text(i - w/2, b + 0.01, '%.3f' % b, ha='center', fontsize=9)
    ax.text(i + w/2, a + 0.01, '%.3f' % a, ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('before_vs_after.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Save & Download Fine-Tuned Model
Download the `fine_tuned_model.zip` and place it in your project folder.

In [ ]:
import shutil

# Save final weights
os.makedirs('fine_tuned_model', exist_ok=True)
torch.save(ecapa.state_dict(), 'fine_tuned_model/embedding_model_final.ckpt')

# Copy config files from pretrained
src = 'pretrained_models/spkrec-ecapa-voxceleb'
for f in os.listdir(src):
    s = os.path.join(src, f)
    d = os.path.join('fine_tuned_model', f)
    if os.path.isfile(s) and f != 'embedding_model.ckpt':
        shutil.copy2(s, d)

# Ensure best model is the one used
best = 'fine_tuned_model/embedding_model.ckpt'
if not os.path.exists(best):
    torch.save(ecapa.state_dict(), best)

# Zip for download
shutil.make_archive('fine_tuned_model', 'zip', '.', 'fine_tuned_model')
print('Model saved and zipped!')
print('Files in fine_tuned_model/:')
for f in sorted(os.listdir('fine_tuned_model')):
    size = os.path.getsize(os.path.join('fine_tuned_model', f))
    print('  %s (%s)' % (f, '%.1f MB' % (size/1e6) if size > 1e6 else '%.1f KB' % (size/1e3)))

In [ ]:
# Download the model
from google.colab import files
files.download('fine_tuned_model.zip')
print('Download started! Save it to your project folder.')

## 11. How to Use in Your Project

After downloading `fine_tuned_model.zip`:

```
1. Unzip into your project:
   speaker_verification_attendence-main/
   └── fine_tuned_model/
       ├── embedding_model.ckpt
       ├── hyperparams.yaml
       └── ...

2. Change 1 line in app/ml_engine/embedding.py:
   BEFORE: source="speechbrain/spkrec-ecapa-voxceleb"
   AFTER:  source="./fine_tuned_model/"

3. Delete old database:
   rm voice_attendance.db

4. Restart server & re-enroll students
```